# Random Forest na Classificação de Presença no ENEM

# Seção 1: Importação do Dataset e Dataset Usado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_parquet("dados_treino.parquet")

# Seção 3: Achando os Melhores Parâmetros

In [ ]:
n_estimators = [int(x) for x in np.linspace(start=50, stop=100, num=6)]
max_features = ['sqrt', 'log2']         
max_depth = [int(x) for x in np.linspace(start=10, stop=40, num=4)]
max_depth.append(None)

param_grid = {
    'n_estimators':      n_estimators,   
    'max_features':      max_features,   
    'max_depth':         max_depth,     
    'min_samples_split': [10, 20, 50],   
    'min_samples_leaf':  [10, 25, 50],   
}

In [ ]:
df_reduzido = df.sample(n=500_000, random_state=42)

In [ ]:
Features = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA',
            "IDADE_ALTA", "ACESSO_DIGITAL", 'PAIS_SEM_INSTRUCAO', 'SEM_DIGITAL', 'FAMILIA_GRANDE_POBRE', 'SCORE_BENS_RELEVANTES',
            'TREINEIRO_JOVEM', 'FAMILIA_QUALIFICADA', 'SCORE_RISCO', 'MORA_SOZINHO', 'PUBLICO_CURSANDO', 'SEM_BENS', 'NU_ANO',
            'TP_ST_CONCLUSAO', 'TP_DEPENDENCIA_ADM_ESC', 'TP_SEXO']


X = df_reduzido[Features]
y = df_reduzido['FALTOU']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTreino: {len(x_train):,} | Teste: {len(x_test):,}")
print(f"Taxa de falta no treino: {y_train.mean():.3f}")
print(f"Taxa de falta no teste:  {y_test.mean():.3f}")

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)

cv_rf = RandomizedSearchCV(   
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='f1_weighted',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

cv_rf.fit(x_train, y_train)

print(cv_rf.best_estimator_)
print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(x_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  cv_rf.predict(x_test))))
print(classification_report(y_test, cv_rf.predict(x_test)))